In this notebook we test the model using the Gradio Interface from HuggingFace, instead of setting up a server, in case you want to test the model right away with no complex setups.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

Installing the dependencies.

In [ ]:
!pip install torch gradio transformers bitsandbytes

Log in to HuggingFace, even though it's not necessary because the model is public.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

Import the model offloading it to a folder, then the tokenizer, and finally the pipeline for inference.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

model = AutoModelForCausalLM.from_pretrained(
    "sse3ele3/VictuAI-v5",
    torch_dtype=torch.float16,
    device_map="auto",
    offload_folder="/kaggle/working/offload"
)

tokenizer = AutoTokenizer.from_pretrained("sse3ele3/VictuAI-v5")

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
    
)

The function to get the model's response. I've already explained in the deployment notebook what the model has been optimized for.

In [ ]:
def predict(input):
    completion = pipe(
        input,
        max_new_tokens=1500,          
        temperature=0.1,               
        do_sample=True,
        top_p=0.95,
        top_k=50,
        repetition_penalty=1.05,       
        length_penalty=1.2,            
        early_stopping=False,          
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        return_full_text=False,
        num_return_sequences=1,
        remove_invalid_values=True
        )[0]["generated_text"]
    return completion

With this test prompt in Qwen chat template format we test the model and get a response.

In [ ]:
test_prompt = """<|im_start|>system
Act as a nutrition expert. Give ONLY the final meal plan. Do not explain your reasoning. Do not think step by step. Start immediately with "Daily Meal Plan:"
<|im_end|>

<|im_start|>user
Age: 30
Gender: Female
Height: 165 cm
Weight: 65 kg
Activity Level: Sedentary
Dietary Preference: Vegetarian
Daily Calorie Target: 1800 kcal
<|im_end|>

<|im_start|>assistant"""

print(predict(test_prompt))

Now, if we want to talk to the model in a Gradio interface, you can do it running this cell.

In [ ]:
import gradio as gr

demo = gr.Interface(fn=predict, inputs="text", outputs="text")

demo.launch()